<a href="https://colab.research.google.com/github/Nanda-Lopes/AlgoStudies/blob/main/Stanford_Algorithms_Specialization_W1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programming Assignment #1


In [2]:
import sys
import threading
from collections import defaultdict

In [3]:
sys.setrecursionlimit(2000000)
threading.stack_size(67108864)

0

In [11]:
filename = '_410e934e6553ac56409b2cb7096a44aa_SCC.txt'

graph = defaultdict(list)
rev_graph = defaultdict(list)
max_vertex = 0

print("1. Lendo o arquivo...")

with open(filename, 'r') as f:
    for line in f:
        parts = line.split()
        if len(parts) >= 2:
            u_str = parts.pop(0)
            v_str = parts.pop(0)

            u = int(u_str)
            v = int(v_str)

            graph[u].append(v)
            rev_graph[v].append(u)

            if u > max_vertex:
                max_vertex = u
            if v > max_vertex:
                max_vertex = v

print(f"Leitura concluída! Vértice máximo encontrado: {max_vertex}")

1. Lendo o arquivo...
Leitura concluída! Vértice máximo encontrado: 875714


Processamento quebrado em partes para evitar colapso or exceso de recursões:
1. Criação de conjunto vazio chamado 'explored', que vai rastrear caminhos já explorados (evita loops infinitos).
2. Cria uma lista vazia 'finish_order, que vai guardar a ordem exata em que os nós terminam de ser processados (ponto importante do algoritmo de Kosaraju).
3. Busca em profundidade através da função 'dfs_rev', que marca o nó atual como explorado, visita os vizinhos no grafo reverso (rev_graph) e adiciona o nó na lista 'finish_order' após não ter para onde ir.
4. Busca em loop de trás para frente através da função 'run_pass_1' do vértice 875714 até 1, adicionando o nó caso ainda não tenha sido explorado.

In [12]:
explored = set()
finish_order = []

def dfs_rev(node):
    explored.add(node)
    for neighbor in rev_graph[node]:
        if neighbor not in explored:
            dfs_rev(neighbor)
    finish_order.append(node)

def run_pass_1():
    print("2. Iniciando Passo 1 (DFS no grafo reverso)...")
    for i in range(max_vertex, 0, -1):
        if i not in explored:
            dfs_rev(i)
    print(f"Passo 1 concluído! Nós processados: {len(finish_order)}")

thread = threading.Thread(target=run_pass_1)
thread.start()
thread.join()

2. Iniciando Passo 1 (DFS no grafo reverso)...
Passo 1 concluído! Nós processados: 875714


Passo **final** para encontrar os componentes e formatar a resposta, ainda impedindo que o Colab trave pelo excesso de recursões da nova busca:
1. Limpeza do conjunto 'explored' (através do explored.clear()), podendo rastrear os caminhos do zero.
2. Busca em profundidade através da função 'dfs_forward', que marca o nó atual como explorado, visita os vizinhos no grafo original (graph) e soma 1 na variável 'count' para cada nó descoberto, calculando o tamanho exato do respectivo grupo.
3. Busca em loop de trás para frente, lendo a lista 'finish_order'. Caso o nó ainda não tenha sido explorado, a busca nele é acionada.
> Ponto importante de Kosaraju: fazer o DFS nesta ordem específica, garantindo que a busca isole e descubra um Strongly Connected Components (SCCs) por vez

4. Formatação da resposta ordena a lista com os tamanhos encontrados do maior para o menor (sort(reverse=True)), pegando os 5 maiores e preenchendo com zeros caso o grafo tenha menos de 5 SCCs no total.


In [13]:
scc_sizes = []

def dfs_forward(node):
    explored.add(node)
    count = 1
    for neighbor in graph[node]:
        if neighbor not in explored:
            count += dfs_forward(neighbor)
    return count

def run_pass_2():
    explored.clear()
    print("3. Iniciando Passo 2 (DFS no grafo original)...")

    for node in reversed(finish_order):
        if node not in explored:
            size = dfs_forward(node)
            scc_sizes.append(size)

    print("4. Formatando a resposta...")
    scc_sizes.sort(reverse=True)

    top_5 = scc_sizes[:5]
    while len(top_5) < 5:
        top_5.append(0)

    print("\n==================================")
    print("RESPOSTA:")
    print(",".join(map(str, top_5)))
    print("==================================\n")

thread = threading.Thread(target=run_pass_2)
thread.start()
thread.join()

3. Iniciando Passo 2 (DFS no grafo original)...
4. Formatando a resposta...

RESPOSTA:
434821,968,459,313,211

